<a href="https://colab.research.google.com/github/Rupeshkumar780/Basics-of-Machine-Learning/blob/main/PCA_Digit_Recognizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

digit_recognizer_path = kagglehub.competition_download('digit-recognizer')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# we will work on train.csv
df = pd.read_csv(os.path.join(digit_recognizer_path, 'train.csv'))
df.head()

In [ ]:
df.shape   # (28 x 28) pixels -> 784 + (1 Label column)
# 42000 images / rows

In [ ]:
df.sample()

In [ ]:
import matplotlib.pyplot as pyplot

pyplot.imshow(df.iloc[29895, 1:].values.reshape(28,28))        # imshow() -> image show
# display the image that is stored on 29895 row, by taking all the values from df and rearranging it (28 x 28) pixels grid

<h2 style='color:red'> Without Using PCA </h2>

In [ ]:
from sklearn.model_selection import train_test_split
X = df.iloc[:, 1:]
y = df.iloc[:, 0]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train.shape, X_test.shape

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()

# in KNN, we calc distance of every particular image with all other images,
# so that we find out the similar images, then we find out the label of the image.

# it will take a lot of time, as in 784 Dim space, we calc the dist of a given point from other 42,000 points,
# then sorting them to find out the neigherest neighbors, then find out their label.

In [ ]:
knn.fit(X_train, y_train)

In [ ]:
import time
start = time.time()
y_pred = knn.predict(X_test)
print(time.time() - start)

In [ ]:
from sklearn.metrics import accuracy_score
print("Accuracy Score : ", accuracy_score(y_pred, y_test))

<h2 style='color:red'> Using PCA </h2>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train.shape, X_test.shape

In [ ]:
# Standardize your Data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train[:3]    # numpy array

In [ ]:
# PCA
from sklearn.decomposition import PCA
# pca = PCA(n_components=None)       # PCA with 784 columns, n_components -> no. of PC you want
pca = PCA(n_components=200)       # PCA with 200 columns (200 Principal Components)

X_train_tnf = pca.fit_transform(X_train)
X_test_tnf = pca.transform(X_test)
X_train_tnf.shape

In [ ]:
knn = KNeighborsClassifier()

knn.fit(X_train_tnf, y_train)
y_pred_tnf = knn.predict(X_test_tnf)

print("Accuracy Score:", accuracy_score(y_pred_tnf, y_test))

In [ ]:
# for i in range(1, 784):
for i in range(1, 20):
    pca = PCA(n_components=i)

    X_train_tnf = pca.fit_transform(X_train)
    X_test_tnf = pca.transform(X_test)

    knn = KNeighborsClassifier()

    knn.fit(X_train_tnf, y_train)
    y_pred_tnf = knn.predict(X_test_tnf)

    print("Accuracy Score:", accuracy_score(y_pred_tnf, y_test))

<h2 style='color:red'> Visualization </h2>

In [ ]:
#  transforming it into 2D corrdinate system
pca = PCA(n_components=2)

X_train_tnf = pca.fit_transform(X_train)
X_test_tnf = pca.transform(X_test)

X_train_tnf

In [ ]:
import plotly.express as px

fig = px.scatter(x = X_train_tnf[:, 0], y=X_train_tnf[:, 1],
                  color = y_train.astype('str'),
                  color_discrete_sequence = px.colors.qualitative.G10
                )
fig.show()

In [ ]:
#  transforming it into 3D corrdinate system
pca = PCA(n_components=3)

X_train_tnf = pca.fit_transform(X_train)
X_test_tnf = pca.transform(X_test)

X_train_tnf

In [ ]:
# eigen values
pca.explained_variance_

In [ ]:
fig = px.scatter_3d(x=X_train_tnf[:, 0], y=X_train_tnf[:, 1], z=X_train_tnf[:, 2],
                    color=y_train.astype('str')
                   )
fig.update_layout(
    margin=dict(l=20, r=20, t=20, b=20)
)

fig.show()

<h2 style="color:red"> Attributes of PCA </h2>

In [ ]:
# Eigen Values
pca.explained_variance_

In [ ]:
# Eigen vectors
print("Eigen Values Shape :", pca.components_.shape)
pca.components_

<h2 style="color:red">Finding Optimum no. of PC </h2>

In [ ]:
# in a 784 Dim Space, it has 784 eigen values

# to find out optimum no. of PC, no. of PC required to explain variance of atleast 90% .
 #             λi
# -------------------------- x 100  ->  explained_variance
#   λ1 + λ2 + ..... + λ784
# every single eigen values tells us their corresponding eigen vector explains how much of variance of original data.

In [ ]:
pca = PCA(n_components=None)

X_train_tnf = pca.fit_transform(X_train)
X_test_tnf = pca.transform(X_test)

In [ ]:
print(pca.explained_variance_.shape)
# pca.explained_variance_

In [ ]:
print(pca.explained_variance_ratio_.shape)          # 784 eigen values
# pca.explained_variance_ratio_

In [ ]:
# 784 eigen vectors in 784 Dim Space
print(pca.components_.shape)
# pca.components_

In [ ]:
# applying Cumulative Sum
cum_value = np.cumsum(pca.explained_variance_ratio_)

# Plot the cumulative sum over a graph
pyplot.plot(cum_value)
pyplot.xlabel("Number of Principal Components (PC's)")
pyplot.ylabel("Explained_Variance")
pyplot.show()

In [ ]:
# from graph, we can get a rough idea, as In order to explain variance around 90% , we need approx 200 PC's.